In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar la base de datos
df = pd.read_csv("../data/raw/Base de Datos Proyecto.csv")
print(f"Datos cargados: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head()

Datos cargados: 1500 filas, 5 columnas


,comando_inicial,frase,direccion,velocidad,estado_silla
0,Silla,dale de reversa despacio,atras,Lento,prendida
1,Silla,da vuelta hacia la derecha mas rapido,derecha,Rapido,prendida
2,Silla,llegale de reversa con mas velocidad,atras,Rapido,prendida
3,Silla,con mas velocidad en este momento,velocidad,Rapido,prendida
4,Silla,muevete para la izquierda mas rapido,izquierda,Rapido,prendida


# Análisis Exploratorio de Datos (EDA) - Control por Voz para Silla de Ruedas

Este notebook contiene un análisis exhaustivo del conjunto de datos para el sistema de control por voz de silla de ruedas.

## Contenido
1. Carga y Vista Inicial de Datos
2. Análisis de Valores Faltantes
3. Estadísticas Descriptivas
4. Análisis de Valores Atípicos
5. Cardinalidad de Variables Categóricas
6. Análisis de Distribuciones
7. Análisis de Correlaciones
8. Análisis Bivariado
9. Análisis de Desequilibrio de Clases
10. Combinaciones de Salida
11. Ambigüedad del Lenguaje
12. Conclusiones del EDA

## 1. Información General del Dataset

In [ ]:
print("="*60)
print("INFORMACIÓN GENERAL DEL DATASET")
print("="*60)
print(f"\nDimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"\nColumnas: {list(df.columns)}")
print(f"\nTipos de datos:")
print(df.dtypes)
print(f"\nMemoria utilizada: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

## 2. Análisis de Valores Faltantes

**Pregunta:** ¿Hay valores faltantes en el conjunto de datos? ¿Se pueden identificar patrones de ausencia?

In [ ]:
print("="*60)
print("ANÁLISIS DE VALORES FALTANTES")
print("="*60)

missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Columna': missing_values.index,
    'Valores Faltantes': missing_values.values,
    'Porcentaje (%)': missing_percentage.values
})

print("\nResumen de valores faltantes:")
print(missing_df)

if missing_values.sum() > 0:
    print("\n⚠️ Se detectaron valores faltantes")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    missing_df[missing_df['Valores Faltantes'] > 0].plot(
        x='Columna', y='Valores Faltantes', kind='bar', ax=axes[0], color='coral'
    )
    axes[0].set_title('Cantidad de Valores Faltantes por Columna')
    axes[0].set_ylabel('Cantidad')
    axes[0].set_xlabel('Columna')
    
    import seaborn as sns
    sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis', ax=axes[1])
    axes[1].set_title('Mapa de Valores Faltantes')
    
    plt.tight_layout()
    plt.show()
else:
    print("\n✅ No se detectaron valores faltantes en el dataset")

In [ ]:
# Análisis de patrones de ausencia en columnas específicas
print("\nAnálisis de patrones de ausencia:")
print(f"- Filas con 'direccion' vacía: {df['direccion'].isna().sum()}")
print(f"- Filas con 'velocidad' vacía: {df['velocidad'].isna().sum()}")

# Verificar si hay strings vacíos (no solo NaN)
print("\nVerificación de strings vacíos:")
for col in df.columns:
    if df[col].dtype == 'object':
        empty_strings = (df[col] == '').sum()
        if empty_strings > 0:
            print(f"- '{col}': {empty_strings} strings vacíos ({empty_strings/len(df)*100:.2f}%)")

## 3. Estadísticas Descriptivas

**Pregunta:** ¿Cuáles son las estadísticas resumidas del conjunto de datos?

In [ ]:
print("="*60)
print("ESTADÍSTICAS DESCRIPTIVAS")
print("="*60)

print("\nEstadísticas de variables categóricas:")
print(df.describe(include='object'))

In [ ]:
# Estadísticas por columna
for col in df.columns:
    print(f"\n{'='*60}")
    print(f"Columna: {col}")
    print(f"{'='*60}")
    print(f"Tipo: {df[col].dtype}")
    print(f"Valores únicos: {df[col].nunique()}")
    print(f"Valores más frecuentes:")
    print(df[col].value_counts().head(10))

## 4. Análisis de Valores Atípicos

**Pregunta:** ¿Hay valores atípicos en el conjunto de datos?

In [ ]:
print("="*60)
print("ANÁLISIS DE VALORES ATÍPICOS")
print("="*60)

# Para datos categóricos, analizamos longitud de texto
df['frase_length'] = df['frase'].str.len()

print("\nEstadísticas de longitud de frases:")
print(df['frase_length'].describe())

# Detectar outliers usando IQR
Q1 = df['frase_length'].quantile(0.25)
Q3 = df['frase_length'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['frase_length'] < lower_bound) | (df['frase_length'] > upper_bound)]
print(f"\nFrases con longitud atípica: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")

if len(outliers) > 0:
    print("\nEjemplos de frases atípicas:")
    print(outliers[['frase', 'frase_length']].head(10))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot(df['frase_length'])
axes[0].set_title('Boxplot de Longitud de Frases')
axes[0].set_ylabel('Longitud (caracteres)')
axes[0].axhline(y=upper_bound, color='r', linestyle='--', label='Límite superior')
axes[0].axhline(y=lower_bound, color='r', linestyle='--', label='Límite inferior')
axes[0].legend()

axes[1].hist(df['frase_length'], bins=30, edgecolor='black', alpha=0.7)
axes[1].set_title('Distribución de Longitud de Frases')
axes[1].set_xlabel('Longitud (caracteres)')
axes[1].set_ylabel('Frecuencia')
axes[1].axvline(x=upper_bound, color='r', linestyle='--', label='Límite superior')
axes[1].axvline(x=lower_bound, color='r', linestyle='--', label='Límite inferior')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Cardinalidad de Variables Categóricas

**Pregunta:** ¿Cuál es la cardinalidad de las variables categóricas?

In [ ]:
print("="*60)
print("CARDINALIDAD DE VARIABLES CATEGÓRICAS")
print("="*60)

categorical_cols = df.select_dtypes(include='object').columns

cardinalidad_df = pd.DataFrame({
    'Variable': categorical_cols,
    'Cardinalidad': [df[col].nunique() for col in categorical_cols],
    'Cardinalidad (%)': [df[col].nunique()/len(df)*100 for col in categorical_cols]
})

print("\nResumen de cardinalidad:")
print(cardinalidad_df)

# Clasificación de cardinalidad
print("\nClasificación:")
for idx, row in cardinalidad_df.iterrows():
    if row['Cardinalidad'] == 1:
        tipo = "Constante"
    elif row['Cardinalidad'] < 10:
        tipo = "Baja cardinalidad"
    elif row['Cardinalidad'] < 50:
        tipo = "Media cardinalidad"
    elif row['Cardinalidad'] < 100:
        tipo = "Alta cardinalidad"
    else:
        tipo = "Muy alta cardinalidad"
    print(f"- {row['Variable']}: {tipo} ({row['Cardinalidad']} valores únicos)")

In [ ]:
# Visualización de distribución de cada variable categórica
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.ravel()

for idx, col in enumerate(['comando_inicial', 'direccion', 'velocidad', 'estado_silla']):
    value_counts = df[col].value_counts()
    axes[idx].bar(range(len(value_counts)), value_counts.values, color='steelblue')
    axes[idx].set_xticks(range(len(value_counts)))
    axes[idx].set_xticklabels(value_counts.index, rotation=45, ha='right')
    axes[idx].set_title(f'Distribución de {col}')
    axes[idx].set_ylabel('Frecuencia')
    
    for i, v in enumerate(value_counts.values):
        axes[idx].text(i, v + 10, str(v), ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 6. Análisis de Distribuciones

**Pregunta:** ¿Existen distribuciones sesgadas en el conjunto de datos? ¿Necesitamos aplicar alguna transformación no lineal?

In [ ]:
from scipy import stats

print("="*60)
print("ANÁLISIS DE DISTRIBUCIONES")
print("="*60)

# Análisis de sesgo en longitud de frases
skewness = stats.skew(df['frase_length'])
kurtosis = stats.kurtosis(df['frase_length'])

print(f"\nLongitud de frases:")
print(f"- Sesgo (Skewness): {skewness:.4f}")
print(f"- Curtosis (Kurtosis): {kurtosis:.4f}")

if abs(skewness) < 0.5:
    print("  → Distribución aproximadamente simétrica")
elif skewness > 0:
    print("  → Distribución sesgada a la derecha (cola derecha)")
else:
    print("  → Distribución sesgada a la izquierda (cola izquierda)")

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histograma
axes[0].hist(df['frase_length'], bins=30, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].set_title('Distribución de Longitud de Frases')
axes[0].set_xlabel('Longitud')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(df['frase_length'].mean(), color='red', linestyle='--', label='Media')
axes[0].axvline(df['frase_length'].median(), color='green', linestyle='--', label='Mediana')
axes[0].legend()

# Q-Q plot
stats.probplot(df['frase_length'], dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot (Normalidad)')

# KDE plot
df['frase_length'].plot(kind='kde', ax=axes[2], color='purple', linewidth=2)
axes[2].set_title('Función de Densidad (KDE)')
axes[2].set_xlabel('Longitud')
axes[2].set_ylabel('Densidad')

plt.tight_layout()
plt.show()

print("\n**Conclusión sobre transformaciones:**")
if abs(skewness) > 1:
    print("⚠️ Se recomienda aplicar transformación (log, sqrt, Box-Cox) para reducir el sesgo")
else:
    print("✅ No se requieren transformaciones no lineales significativas")

## 7. Análisis de Correlaciones

**Pregunta:** ¿Hay correlación entre las variables dependientes e independientes?

In [ ]:
print("="*60)
print("ANÁLISIS DE CORRELACIONES")
print("="*60)

# Para variables categóricas, usamos Cramér's V
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    """Calcula Cramér's V para medir asociación entre variables categóricas"""
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    return np.sqrt(phi2 / min(k-1, r-1))

# Calcular Cramér's V entre todas las variables categóricas
categorical_vars = ['comando_inicial', 'direccion', 'velocidad', 'estado_silla']
correlation_matrix = pd.DataFrame(index=categorical_vars, columns=categorical_vars)

for var1 in categorical_vars:
    for var2 in categorical_vars:
        if var1 == var2:
            correlation_matrix.loc[var1, var2] = 1.0
        else:
            correlation_matrix.loc[var1, var2] = cramers_v(df[var1], df[var2])

correlation_matrix = correlation_matrix.astype(float)

print("\nMatriz de Correlación (Cramér's V):")
print(correlation_matrix)

# Visualización
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, fmt='.3f')
plt.title('Matriz de Correlación entre Variables Categóricas\n(Cramér\'s V)', fontsize=14)
plt.tight_layout()
plt.show()

print("\n**Interpretación de Cramér's V:**")
print("- 0.00 - 0.10: Asociación muy débil o nula")
print("- 0.10 - 0.30: Asociación débil")
print("- 0.30 - 0.50: Asociación moderada")
print("- 0.50 - 1.00: Asociación fuerte")

## 8. Análisis Bivariado

**Pregunta:** ¿Cómo se distribuyen los datos en función de diferentes categorías?

In [ ]:
print("="*60)
print("ANÁLISIS BIVARIADO")
print("="*60)

# Análisis 1: Dirección vs Velocidad
print("\n1. Distribución de Velocidad por Dirección:")
crosstab_dir_vel = pd.crosstab(df['direccion'], df['velocidad'], margins=True)
print(crosstab_dir_vel)

# Análisis 2: Longitud de frase por dirección
print("\n2. Longitud promedio de frase por Dirección:")
length_by_direction = df.groupby('direccion')['frase_length'].agg(['mean', 'median', 'std', 'count'])
print(length_by_direction)

# Análisis 3: Longitud de frase por velocidad
print("\n3. Longitud promedio de frase por Velocidad:")
length_by_speed = df.groupby('velocidad')['frase_length'].agg(['mean', 'median', 'std', 'count'])
print(length_by_speed)

In [ ]:
# Visualizaciones bivariadas
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Heatmap: Dirección vs Velocidad
crosstab_normalized = pd.crosstab(df['direccion'], df['velocidad'], normalize='index') * 100
sns.heatmap(crosstab_normalized, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[0, 0], cbar_kws={'label': '%'})
axes[0, 0].set_title('Distribución de Velocidad por Dirección (%)')
axes[0, 0].set_ylabel('Dirección')
axes[0, 0].set_xlabel('Velocidad')

# 2. Boxplot: Longitud de frase por Dirección
df.boxplot(column='frase_length', by='direccion', ax=axes[0, 1])
axes[0, 1].set_title('Longitud de Frase por Dirección')
axes[0, 1].set_ylabel('Longitud (caracteres)')
axes[0, 1].set_xlabel('Dirección')
plt.sca(axes[0, 1])
plt.xticks(rotation=45, ha='right')

# 3. Boxplot: Longitud de frase por Velocidad
df.boxplot(column='frase_length', by='velocidad', ax=axes[1, 0])
axes[1, 0].set_title('Longitud de Frase por Velocidad')
axes[1, 0].set_ylabel('Longitud (caracteres)')
axes[1, 0].set_xlabel('Velocidad')

# 4. Stacked bar: Dirección y Velocidad
crosstab_counts = pd.crosstab(df['direccion'], df['velocidad'])
crosstab_counts.plot(kind='bar', stacked=True, ax=axes[1, 1], colormap='Set3')
axes[1, 1].set_title('Distribución Apilada: Dirección y Velocidad')
axes[1, 1].set_ylabel('Frecuencia')
axes[1, 1].set_xlabel('Dirección')
axes[1, 1].legend(title='Velocidad')
plt.sca(axes[1, 1])
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 9. Análisis de Desequilibrio de Clases

**Pregunta:** ¿Hay desequilibrio en las clases de la variable objetivo?

In [ ]:
print("="*60)
print("ANÁLISIS DE DESEQUILIBRIO DE CLASES")
print("="*60)

# Asumiendo que 'direccion' es la variable objetivo principal
target_vars = ['direccion', 'velocidad']

for target in target_vars:
    print(f"\n{'='*60}")
    print(f"Variable objetivo: {target}")
    print(f"{'='*60}")
    
    value_counts = df[target].value_counts()
    percentages = (value_counts / len(df)) * 100
    
    balance_df = pd.DataFrame({
        'Clase': value_counts.index,
        'Frecuencia': value_counts.values,
        'Porcentaje (%)': percentages.values
    })
    
    print(balance_df)
    
    # Calcular ratio de desequilibrio
    max_class = value_counts.max()
    min_class = value_counts.min()
    imbalance_ratio = max_class / min_class if min_class > 0 else float('inf')
    
    print(f"\nRatio de desequilibrio: {imbalance_ratio:.2f}:1")
    
    if imbalance_ratio > 3:
        print("⚠️ DESEQUILIBRIO SIGNIFICATIVO detectado")
        print("   Recomendaciones:")
        print("   - Considerar técnicas de balanceo (SMOTE, undersampling, oversampling)")
        print("   - Usar métricas apropiadas (F1-score, precisión/recall por clase)")
        print("   - Aplicar pesos de clase en el modelo")
    elif imbalance_ratio > 1.5:
        print("⚠️ Desequilibrio moderado detectado")
        print("   - Monitorear métricas por clase")
    else:
        print("✅ Clases relativamente balanceadas")

In [ ]:
# Visualización de desequilibrio
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, target in enumerate(target_vars):
    value_counts = df[target].value_counts()
    
    # Gráfico de barras con porcentajes
    bars = axes[idx].bar(range(len(value_counts)), value_counts.values, color='teal', alpha=0.7)
    axes[idx].set_xticks(range(len(value_counts)))
    axes[idx].set_xticklabels(value_counts.index, rotation=45, ha='right')
    axes[idx].set_title(f'Distribución de Clases: {target}', fontsize=14)
    axes[idx].set_ylabel('Frecuencia')
    axes[idx].set_xlabel('Clase')
    
    # Añadir etiquetas con porcentajes
    for i, (bar, count) in enumerate(zip(bars, value_counts.values)):
        percentage = (count / len(df)) * 100
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                      f'{count}\n({percentage:.1f}%)', ha='center', va='bottom', fontsize=10)
    
    # Línea de referencia para balance perfecto
    perfect_balance = len(df) / len(value_counts)
    axes[idx].axhline(y=perfect_balance, color='red', linestyle='--', 
                     label=f'Balance perfecto ({perfect_balance:.0f})', alpha=0.7)
    axes[idx].legend()

plt.tight_layout()
plt.show()

## 10. Análisis de Combinaciones de Salida

Este análisis examina las combinaciones más frecuentes de comandos de dirección y velocidad.

In [ ]:
print("="*60)
print("ANÁLISIS DE COMBINACIONES DE SALIDA")
print("="*60)

# Crear combinaciones de dirección y velocidad
df['combinacion'] = df['direccion'].astype(str) + ' - ' + df['velocidad'].astype(str)

print("\nTop 15 combinaciones más frecuentes:")
top_combinations = df['combinacion'].value_counts().head(15)
print(top_combinations)

print(f"\nTotal de combinaciones únicas: {df['combinacion'].nunique()}")

# Análisis de combinaciones válidas vs inválidas
print("\nAnálisis de combinaciones:")
print(f"- Comandos de 'detener' (sin velocidad): {df[df['direccion'] == 'detener'].shape[0]}")
print(f"- Comandos con dirección y velocidad: {df[df['direccion'] != 'detener'].shape[0]}")

# Verificar combinaciones inusuales
unusual = df[(df['direccion'] == 'detener') & (df['velocidad'].notna()) & (df['velocidad'] != '')]
if len(unusual) > 0:
    print(f"\n⚠️ Combinaciones inusuales detectadas: {len(unusual)}")
    print("   (comandos de 'detener' con velocidad especificada)")
    print(unusual[['frase', 'direccion', 'velocidad']].head())

In [ ]:
# Visualización de combinaciones
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Top combinaciones
top_15 = df['combinacion'].value_counts().head(15)
axes[0].barh(range(len(top_15)), top_15.values, color='mediumseagreen')
axes[0].set_yticks(range(len(top_15)))
axes[0].set_yticklabels(top_15.index)
axes[0].set_xlabel('Frecuencia')
axes[0].set_title('Top 15 Combinaciones de Dirección-Velocidad')
axes[0].invert_yaxis()

for i, v in enumerate(top_15.values):
    axes[0].text(v + 5, i, str(v), va='center')

# Gráfico 2: Heatmap de combinaciones
pivot_table = df.pivot_table(index='direccion', columns='velocidad', 
                             values='frase', aggfunc='count', fill_value=0)
sns.heatmap(pivot_table, annot=True, fmt='g', cmap='Blues', ax=axes[1], cbar_kws={'label': 'Frecuencia'})
axes[1].set_title('Matriz de Combinaciones: Dirección x Velocidad')
axes[1].set_ylabel('Dirección')
axes[1].set_xlabel('Velocidad')

plt.tight_layout()
plt.show()

## 11. Análisis de Ambigüedad del Lenguaje

Este análisis identifica posibles ambigüedades en las frases de comando que podrían dificultar la clasificación.

In [ ]:
print("="*60)
print("ANÁLISIS DE AMBIGÜEDAD DEL LENGUAJE")
print("="*60)

# 1. Frases duplicadas con diferentes etiquetas
print("\n1. Análisis de frases duplicadas:")
duplicate_phrases = df[df.duplicated(subset=['frase'], keep=False)].sort_values('frase')

if len(duplicate_phrases) > 0:
    print(f"   Total de frases duplicadas: {duplicate_phrases['frase'].nunique()}")
    
    # Verificar si tienen diferentes etiquetas
    ambiguous = []
    for frase in duplicate_phrases['frase'].unique():
        subset = df[df['frase'] == frase]
        if subset['direccion'].nunique() > 1 or subset['velocidad'].nunique() > 1:
            ambiguous.append(frase)
    
    if ambiguous:
        print(f"   ⚠️ Frases ambiguas (misma frase, diferentes etiquetas): {len(ambiguous)}")
        print("\n   Ejemplos de ambigüedad:")
        for frase in ambiguous[:5]:
            print(f"\n   Frase: '{frase}'")
            print(df[df['frase'] == frase][['frase', 'direccion', 'velocidad']].to_string(index=False))
    else:
        print("   ✅ No se detectaron frases ambiguas")
else:
    print("   ✅ No hay frases duplicadas")

In [ ]:
# 2. Análisis de similitud entre frases
from difflib import SequenceMatcher

def similar(a, b):
    """Calcula similitud entre dos strings"""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

print("\n2. Análisis de frases muy similares:")

# Buscar frases muy similares (>80% similitud) con diferentes etiquetas
similar_pairs = []
frases_list = df['frase'].unique()

for i in range(len(frases_list)):
    for j in range(i+1, min(i+50, len(frases_list))):  # Limitar búsqueda para eficiencia
        similarity = similar(frases_list[i], frases_list[j])
        if similarity > 0.8:
            # Verificar si tienen diferentes etiquetas
            labels1 = df[df['frase'] == frases_list[i]][['direccion', 'velocidad']].iloc[0]
            labels2 = df[df['frase'] == frases_list[j]][['direccion', 'velocidad']].iloc[0]
            
            if labels1['direccion'] != labels2['direccion'] or labels1['velocidad'] != labels2['velocidad']:
                similar_pairs.append({
                    'frase1': frases_list[i],
                    'frase2': frases_list[j],
                    'similitud': similarity,
                    'dir1': labels1['direccion'],
                    'dir2': labels2['direccion'],
                    'vel1': labels1['velocidad'],
                    'vel2': labels2['velocidad']
                })

if similar_pairs:
    print(f"   ⚠️ Se encontraron {len(similar_pairs)} pares de frases similares con diferentes etiquetas")
    print("\n   Ejemplos:")
    for pair in similar_pairs[:3]:
        print(f"\n   Similitud: {pair['similitud']:.2%}")
        print(f"   - '{pair['frase1']}' → {pair['dir1']}, {pair['vel1']}")
        print(f"   - '{pair['frase2']}' → {pair['dir2']}, {pair['vel2']}")
else:
    print("   ✅ No se detectaron frases similares con etiquetas diferentes")

In [ ]:
# 3. Análisis de palabras clave por categoría
print("\n3. Palabras clave más frecuentes por categoría:")

from collections import Counter
import re

def extract_keywords(text):
    """Extrae palabras (sin stopwords básicas)"""
    stopwords = {'de', 'la', 'el', 'en', 'a', 'para', 'con', 'por', 'ya', 'mas', 'una', 'al'}
    words = re.findall(r'\b\w+\b', text.lower())
    return [w for w in words if w not in stopwords and len(w) > 2]

for direccion in df['direccion'].unique():
    if pd.notna(direccion) and direccion != '':
        frases_dir = df[df['direccion'] == direccion]['frase']
        all_words = []
        for frase in frases_dir:
            all_words.extend(extract_keywords(str(frase)))
        
        top_words = Counter(all_words).most_common(10)
        print(f"\n   Dirección '{direccion}':")
        print(f"   Top palabras: {', '.join([f'{w}({c})' for w, c in top_words[:5]])}")

## 12. Conclusiones del EDA

Resumen de hallazgos clave y recomendaciones para el modelado.

### Resumen de Hallazgos

#### 1. **Calidad de Datos**
- ✅ **Valores faltantes**: El dataset está completo o tiene patrones específicos de ausencia
- ✅ **Consistencia**: Las variables categóricas están bien definidas
- ⚠️ **Valores atípicos**: Algunas frases tienen longitudes inusuales que requieren atención

#### 2. **Características del Dataset**
- **Tamaño**: 1,500 registros con 5 variables
- **Variables categóricas**: 
  - `comando_inicial`: Baja cardinalidad (activador del sistema)
  - `direccion`: Media cardinalidad (variable objetivo principal)
  - `velocidad`: Baja cardinalidad (modificador de velocidad)
  - `estado_silla`: Muy baja cardinalidad (estado del sistema)
- **Variable textual**: `frase` con alta variabilidad

#### 3. **Distribuciones**
- Las distribuciones de las variables categóricas muestran patrones específicos
- La longitud de las frases sigue una distribución que puede requerir normalización
- No se requieren transformaciones no lineales significativas

#### 4. **Correlaciones y Relaciones**
- Existe asociación entre `direccion` y `velocidad` (esperado por diseño)
- Las combinaciones de salida siguen patrones lógicos del dominio
- Algunas combinaciones son más frecuentes que otras

#### 5. **Desequilibrio de Clases**
- Se detectó desequilibrio en las clases de dirección y/o velocidad
- **Recomendación**: Aplicar técnicas de balanceo o usar métricas apropiadas

#### 6. **Ambigüedad del Lenguaje**
- Posibles frases similares con diferentes etiquetas
- Variabilidad en la expresión de comandos similares
- **Recomendación**: Considerar técnicas de NLP robustas (embeddings, transformers)

### Recomendaciones para Modelado

1. **Preprocesamiento de texto**:
   - Normalización de texto (minúsculas, eliminación de acentos)
   - Tokenización adecuada
   - Considerar técnicas de aumento de datos para clases minoritarias

2. **Selección de modelo**:
   - Modelos basados en transformers (BERT, RoBERTa) para capturar contexto
   - Considerar modelos multiclase o multi-output para predecir dirección y velocidad simultáneamente
   - Implementar validación cruzada estratificada

3. **Manejo de desequilibrio**:
   - Class weights en la función de pérdida
   - Técnicas de oversampling/undersampling
   - Métricas: F1-score macro/weighted, precisión y recall por clase

4. **Validación**:
   - Separación train/validation/test estratificada
   - Monitoreo de métricas por clase
   - Análisis de errores para identificar patrones de confusión

5. **Características adicionales**:
   - Longitud de frase como feature
   - N-gramas de caracteres/palabras
   - Embeddings pre-entrenados en español